# Sesión 7 · RAG avanzado: técnicas, GraphRAG y evaluación

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Explicar por qué falla un RAG básico** con preguntas que necesitan varios fragmentos (multi-sección) o varios pasos de razonamiento (multi-salto), y **medirlo**.
2. **Aplicar técnicas avanzadas de recuperación**: *small-to-big* (buscar con lo pequeño, entregar lo grande), encabezados contextuales, descomposición de preguntas y HyDE.
3. **Explicar GraphRAG**: qué es un grafo de conocimiento, cuándo aporta (preguntas relacionales y globales) y cuándo **no** compensa; y compararlo **con números** frente a la búsqueda vectorial.
4. **Evaluar un RAG por capas**: Recall@K, Precision@K y MRR para la recuperación; dato correcto, citas válidas, "no sé" y cifras con respaldo para la generación; un **juez LLM** calibrado contra etiquetas humanas.
5. **Reconocer las causas de las alucinaciones, mitigarlas y comprobar la mejora** con el ciclo *medir → cambiar → volver a medir*.

> 🔁 **Lo que NO repetimos.** La reescritura de la consulta, la búsqueda híbrida (palabras clave + semántica) y el reranking son de la Sesión 6. Aquí los damos por sabidos y nos centramos en lo que viene después de ellos.

> ⚠️ **Sigue sin ser un agente.** Todo lo de hoy son flujos fijos. Al final tocamos el *RAG agéntico* solo como adelanto conceptual: lo construirás cuando veas agentes.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | Medir antes de mejorar: golden set y métricas de recuperación | No |
| 2 | Dónde falla el RAG básico | No |
| 3 | *Small-to-big* y encabezados contextuales | No |
| 4 | Descomposición de preguntas y HyDE | Opcional: centavos (gratis con modelo local) |
| 5 | **GraphRAG**: qué es y cuándo conviene | No |
| 6 | GraphRAG medido: vectorial vs grafo vs combinado | Opcional: centavos (gratis con modelo local) |
| 7 | Evaluar la generación y diagnosticar por capa | Opcional: centavos (gratis con modelo local) |
| 8 | Alucinaciones: causas, mitigación y regresión | Opcional: centavos (gratis con modelo local) |
| 9 | El juez LLM: calibrarlo y conocer sus sesgos | Opcional: centavos (gratis con modelo local) |
| 10 | Práctica integrada: el **Medidor de calidad RAG** | Opcional |
| 11 | Cierre | No |

> 💰 Las celdas que llaman al LLM llevan **💰 FYI** y una estimación. Con un modelo de proveedor cada pregunta cuesta ≈ US$ 0.0005; con el modelo local de LM Studio, 0. Las métricas de recuperación **nunca** llaman al LLM: son gratis.

> 📝 Cada parte alterna explicación y código. Lee la explicación **antes** de ejecutar la celda y compara lo que ves con lo que se esperaba.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s07-rag-avanzado y selecciona el kernel .venv (ver README)."
)

In [ ]:
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from almacenes import Fragmento, crear_almacen
from avanzado import (
    RecuperadorDescompuesto, RecuperadorHyDE, RecuperadorSmallToBig, descomponer, fusionar_rrf,
    ingestar_con_metadatos, ingestar_pequenos,
)
from cliente_llm import Gasto, hay_api_key, local_disponible, modelo_local, variable_api_key
from embeddings import embeddings_disponibles, modelo_embeddings, proveedor_embeddings, vector_consulta, vectorizar
from evaluacion import (
    CAPA_DEL_DIAGNOSTICO, SLO_POR_DEFECTO, calibrar_juez, cargar_golden, con_respuesta, contexto_desde_secciones,
    contiene_claves, evaluar_recuperacion_en_k, evaluar_rag, guardar_informe, imprimir_resumen, informe_markdown,
    juzgar_fidelidad, metricas_recuperacion, modelo_juez, numeros_sin_respaldo, proveedor_juez, puntajes_mejor_fragmento,
    tabla_comparativa, tabla_por_tipo, verificar_slo,
)
from grafo import (
    RecuperadorCombinado, RecuperadorGrafo, afectados_por, administrador_de, cargar_grafo, comparar_tripletas,
    dependencias_de, dibujar_grafo, entidades_en, equipos_de, evaluar_multisalto, extraer_tripletas,
    grafo_desde_tripletas, mas_dependidos, resumen_multisalto, vecindad,
)
from rag import (
    NO_SE, construir_prompt, generar, ingestar, leer_documento, proveedor_generacion, proveedores_disponibles,
    recuperar, texto_a_vectorizar,
)

HAY_EMB, detalle = embeddings_disponibles()
print(("✅ Embeddings: " if HAY_EMB else "❌ Embeddings: ") + detalle)

### 🔌 Quién genera las respuestas

El **LLM** que redacta las respuestas se elige con `RAG_PROVIDER` (o `LLM_PROVIDER`) en tu `.env`: `anthropic`, `openai` o `local` (LM Studio, gratis). En esta sesión **el modelo local alcanza para todo**: lo importante es el método de medir, no la nota que saque tu modelo. Con un modelo de proveedor los números mejoran; compáralos.

> El **juez** (Parte 9) se elige con `JUEZ_PROVIDER` y `JUEZ_MODEL` (opcionales). Por defecto juzga el mismo proveedor que genera.

In [ ]:
GEN = proveedor_generacion()
if GEN == "local":
    HAY_LLM, msg = local_disponible(modelo_local())
else:
    HAY_LLM, msg = hay_api_key(GEN), f"falta {variable_api_key(GEN)} en tu .env"
print(f"Generación con: {GEN} · {'✅ disponible' if HAY_LLM else '❌ ' + str(msg)}")
print(f"Juez: {proveedor_juez()} · modelo {modelo_juez()}")

### 🔌 Interruptor general de gasto

💰 **FYI:** con `False`, ninguna celda llama al LLM (las de recuperación siguen funcionando, son gratis). Con `True` y un proveedor disponible, se ejecutan. Si usas un proveedor de pago, configura antes un límite de gasto mensual (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no llamar al LLM
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_LLM and HAY_EMB
PUEDE_INDEXAR = HAY_EMB
print("Celdas con LLM activas:", PUEDE_GASTAR)

MANUAL = "datos/manual_soporte.md"
GOLDEN = cargar_golden("datos/golden_set.json")
INFORMES = {}          # nombre → Informe, para comparar al final
gasto = Gasto()        # llamadas sueltas; las evaluaciones guardan su propio costo en cada Informe

def correr(nombre, almacen, **opciones):
    # Evalúa el RAG con el golden set completo y guarda el informe con un nombre.
    inf = evaluar_rag(almacen, GOLDEN, **opciones)
    INFORMES[nombre] = inf
    return inf

---
# Parte 1 · Medir antes de mejorar

## Una analogía de infraestructura

Nadie sube un cambio a producción sin dashboards. En un RAG pasa igual:

| Mundo de infraestructura | Mundo del RAG |
| --- | --- |
| **SLI** (indicador): qué se mide, p. ej. latencia p95 | **Métrica**: Recall@K, datos correctos, alucinaciones… |
| **SLO** (objetivo): el mínimo aceptable, p. ej. 99.9 % | **Umbral de aprobación**: "Recall ≥ 90 %, alucinaciones ≤ 10 %" |
| **Prueba de regresión** al desplegar | **Volver a correr el golden set** al cambiar el prompt, el chunking o el modelo |
| Observabilidad: ¿qué capa falló? | **Diagnóstico por capa**: recuperación vs generación |

Sin medir, cada "mejora" es una opinión. Con un conjunto de preguntas fijo, es un experimento.

## El *golden set* (conjunto de oro)

Es una lista de preguntas con la respuesta correcta **escrita por una persona**, más qué parte del documento la contiene. Es la vara con la que se mide todo lo demás. El de hoy está en `datos/golden_set.json`, tiene **24 preguntas de 6 tipos** y usa el manual de Andes Fibra de la Sesión 5.

| Tipo | Qué es | Qué esperamos del RAG |
| --- | --- | --- |
| `directa` | La respuesta está en un solo lugar | Responder con cita |
| `multi_seccion` | Hacen falta datos de **dos secciones** | Traer ambas y combinarlas |
| `multi_salto` | Hay que **encadenar** un dato con otro (y a veces calcular) | Traer ambas y razonar |
| `ambigua` | Puede entenderse de dos maneras | Cubrir las interpretaciones |
| `fuera_parecida` | Suena a algo del manual, pero **no está** | Decir "no sé" (la trampa) |
| `fuera_ajena` | Nada que ver con el manual | Decir "no sé" |

In [ ]:
from collections import Counter

print(f"{len(GOLDEN)} preguntas · {sum(1 for p in GOLDEN if p['rapido'])} marcadas como 'rápidas' (subconjunto de 10 para evaluar en segundos)\n")
for tipo, n in Counter(p["tipo"] for p in GOLDEN).items():
    print(f"  {tipo:<16}{n:>3}")

print("\nAnatomía de una pregunta:")
ejemplo = next(p for p in GOLDEN if p["id"] == "MS02")
for campo in ("id", "tipo", "pregunta", "respuesta_esperada", "secciones", "claves"):
    print(f"  {campo:<19}{ejemplo[campo]}")

Fíjate en `secciones` y `claves`: la primera dice **dónde** está la respuesta (para medir la **recuperación**) y la segunda **qué dato** debe aparecer (para medir la **generación**). Dos preguntas distintas, dos medidas distintas.

## Las tres métricas de recuperación

Cada fragmento recuperado se marca como **relevante** si pertenece a una de las secciones correctas. Con eso:

| Métrica | Pregunta que responde | Cómo se calcula |
| --- | --- | --- |
| **Recall@K** | ¿Traje **todo** lo que necesitaba? | secciones relevantes que aparecen en el top-K ÷ secciones relevantes que existen |
| **Precision@K** | ¿Cuánto de lo que traje **sirve**? | fragmentos relevantes del top-K ÷ K |
| **MRR** | ¿Qué tan **arriba** quedó el primero útil? | 1 ÷ posición del primer fragmento relevante (0 si no aparece) |

Recall sube al traer más fragmentos; Precision baja (traes más ruido). Casi toda decisión de recuperación es un balance entre las dos: el ruido cuesta tokens, dinero y a veces confunde al modelo.

> 🧮 Con **una sola** sección relevante, Recall@K es lo mismo que *hit@K* de la Sesión 5. Con dos o más (las multi-sección), distingue "traje una" de "traje las dos".

### Un ejemplo calculado a mano

In [ ]:
base = crear_almacen("memoria")
ingestar(base, MANUAL, estrategia="oraciones", max_palabras=70, solape=1)     # el RAG de la Sesión 5
print(f"Base: {base.contar()} fragmentos de ~70 palabras\n")

p = next(x for x in GOLDEN if x["id"] == "M01")
print("Pregunta:", p["pregunta"])
print("Secciones relevantes:", p["secciones"], "\n")
top = recuperar(base, p["pregunta"], k=4)
for i, r in enumerate(top, start=1):
    marca = "✅ relevante" if r.metadatos["seccion"] in p["secciones"] else "·  ruido"
    print(f"  {i}. {r.puntaje:.3f}  {marca:<13} {r.metadatos['seccion']}")

m = metricas_recuperacion(top, p)
print(f"\nRecall@4 = {m['recall']:.2f}  (cubre {int(m['recall'] * len(p['secciones']))} de {len(p['secciones'])} secciones)")
print(f"Precision@4 = {m['precision']:.2f}  ({round(m['precision'] * 4)} de 4 fragmentos sirven)")
print(f"MRR = {m['mrr']:.2f}  (el primero útil quedó en la posición {m['primer_relevante']})")

**Por qué etiquetamos por sección y no por fragmento:** si cambias el tamaño de los fragmentos (lo haremos varias veces hoy), los identificadores cambian pero las secciones no. La misma etiqueta sirve para comparar configuraciones distintas.

**Una limitación honesta:** con 24 preguntas, una diferencia de 1 pregunta (≈ 4 puntos) es ruido. El golden set sirve para **comparar ideas**, no para prometer exactitud; uno real necesita decenas o cientos de preguntas reales de usuarios.

---
# Parte 2 · Dónde falla el RAG básico

Medimos el RAG de la Sesión 5 (fragmentos de ~70 palabras, k = 4) **solo en recuperación** (gratis) y lo desglosamos **por tipo de pregunta**. Los promedios globales esconden justo lo interesante.

In [ ]:
inf_base_rec = evaluar_rag(base, GOLDEN, k=4, con_generacion=False)
print(tabla_por_tipo(inf_base_rec))
res = inf_base_rec.resumen()
print(f"\nTodas · Recall@4 = {res['recall']:.0%} · Precision@4 = {res['precision']:.0%} · MRR = {res['mrr']:.0%} · "
      f"{res['palabras_contexto']:.0f} palabras de contexto por pregunta")

In [ ]:
print("Preguntas donde NO llegó toda la información necesaria:\n")
for f in inf_base_rec.filas:
    if f.recall is not None and f.recall < 1:
        p = next(x for x in GOLDEN if x["id"] == f.id)
        print(f"{f.id} ({f.tipo}) recall={f.recall:.2f}")
        print(f"   pregunta : {f.pregunta}")
        print(f"   esperadas: {p['secciones']}")
        print(f"   llegaron : {sorted(set(f.secciones_recuperadas))}\n")

### Lo que se ve

- Las preguntas **directas** casi siempre salen bien: un buen RAG básico hace bien lo básico.
- Las que necesitan **dos temas** (multi-sección y, sobre todo, multi-salto) son las que pierden información: el vector de una pregunta larga se parece **un poco a cada tema y mucho a ninguno**, y los fragmentos del segundo tema quedan fuera del top-4.
- **Subir `k` ayuda, pero no es gratis.** Veámoslo con la curva clásica.

In [ ]:
curva = evaluar_recuperacion_en_k(base, GOLDEN, ks=(1, 2, 3, 4, 6, 8))
ks = list(curva)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(ks, [curva[k]["recall"] for k in ks], "o-", label="Recall@K")
ax.plot(ks, [curva[k]["precision"] for k in ks], "s-", label="Precision@K")
ax.set_xlabel("K (fragmentos entregados al modelo)"); ax.set_ylabel("proporción"); ax.set_ylim(0, 1.05)
ax.set_title("Más K: más cobertura, más ruido"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
for k in ks:
    print(f"K={k}: Recall {curva[k]['recall']:.0%} · Precision {curva[k]['precision']:.0%} · MRR {curva[k]['mrr']:.0%}")

Las dos curvas se cruzan en sentido contrario: **la forma de subir Recall sin hundir Precision es ser más listo, no traer más**. De eso tratan las técnicas que siguen.

> 💬 **Para pensar:** si Recall@8 llega casi a 100 %, ¿por qué no usar siempre K = 8? Piensa en tokens, costo, latencia y en lo que hace un modelo pequeño cuando recibe ocho fragmentos y solo dos sirven.

---
# Parte 3 · *Small-to-big* y encabezados contextuales

## El dilema del tamaño del fragmento

| Fragmento | Ventaja | Problema |
| --- | --- | --- |
| **Pequeño** (≈ 35 palabras) | Su vector habla de **una sola cosa**: se recupera con precisión | Al modelo le falta contexto para responder |
| **Grande** (una sección) | Tiene contexto completo | Su vector se **diluye**; entra ruido al prompt |

## La idea: no elegir

```
 BUSCAR (precisión)                         ENTREGAR (contexto)
 fragmento pequeño  ───►  su "padre":  la sección completa     (modo "seccion")
                                       o él + sus vecinos      (modo "ventana")
```

Se **indexan** los fragmentos pequeños, pero al modelo se le **entrega** el trozo grande al que pertenecen (el *parent document*, "documento padre"). Es el truco detrás de muchos retrievers "parent-document" y "sentence-window".

## Encabezados contextuales

Un fragmento suelto ("Cuesta 60 dólares.") es ambiguo. Anteponerle su **encabezado** (la sección, el título del documento) antes de vectorizarlo le da el tema. Es barato y no cambia lo que se entrega. En nuestro `rag.py` ya se antepone la sección (`texto_a_vectorizar`); aquí **medimos** si importa, quitándola.

> Ambas técnicas son gratis: solo usan embeddings locales.

In [ ]:
# Fragmentos pequeños (35 palabras), con y sin encabezado contextual
peq = crear_almacen("memoria"); frs_peq = ingestar_pequenos(peq, MANUAL, max_palabras=35, encabezado=True)
peq_sin = crear_almacen("memoria"); ingestar_pequenos(peq_sin, MANUAL, max_palabras=35, encabezado=False)
print(f"{len(frs_peq)} fragmentos pequeños (frente a {base.contar()} de ~70 palabras)\n")

informes = {
    "70 pal. k=4": inf_base_rec,
    "35 pal. k=4": evaluar_rag(peq, GOLDEN, k=4, con_generacion=False),
    "35 pal. sin enc.": evaluar_rag(peq_sin, GOLDEN, k=4, con_generacion=False),
}
print(tabla_comparativa(informes, ("recall", "precision", "mrr")))

Fíjate en dos cosas: los fragmentos pequeños mejoran **Precision** (cada fragmento habla de lo suyo), y quitar el encabezado casi no cambia el resultado **en este manual**, porque las secciones ya son temáticas y el texto de cada fragmento nombra el tema. En documentos donde los fragmentos son genéricos ("Ver tabla anterior"), el encabezado pesa mucho más. **Medir te dice cuándo una técnica importa y cuándo es adorno.**

### Ahora, *small-to-big*: buscar pequeño, entregar grande

In [ ]:
informes_s2b = {"base 70 pal. k=4": inf_base_rec}
for modo in ("seccion", "ventana"):
    rec = RecuperadorSmallToBig(peq, MANUAL, frs_peq, modo=modo)
    for k in (2, 3):
        informes_s2b[f"{modo} k={k}"] = evaluar_rag(None, GOLDEN, k=k, con_generacion=False, recuperador=rec)
print(tabla_comparativa(informes_s2b, ("recall", "precision", "mrr")))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for nombre, inf in informes_s2b.items():
    r = inf.resumen()
    ax.scatter(r["palabras_contexto"], r["recall"], s=70)
    ax.annotate(nombre, (r["palabras_contexto"], r["recall"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("palabras de contexto por pregunta (≈ costo)"); ax.set_ylabel("Recall"); ax.grid(alpha=0.3)
ax.set_title("Cobertura vs tamaño del contexto"); plt.show()

### Cómo leer el resultado

- **`seccion`** entrega secciones completas: el Recall sube (el modelo recibe el contexto entero) pero **la Precision baja** y el contexto crece. Es la opción cuando las respuestas dependen de condiciones que están en el resto de la sección.
- **`ventana`** entrega el fragmento y sus vecinos: contexto compacto y buena Precision; conviene cuando el vecino inmediato basta.
- **No hay ganador universal**: depende de cuánto contexto necesite tu respuesta y de cuánto estés dispuesto a pagar en tokens. Por eso se mide con **tus** preguntas.

> ⚠️ Small-to-big **no arregla** lo que está en dos secciones distintas si ninguna de las dos aparece en el top-K de hijos. Para eso, la siguiente técnica.

---
# Parte 4 · Descomposición de preguntas y HyDE

## Preguntas multi-salto

> *"Mi factura venció y quiero pagar en efectivo en una agencia un sábado, ¿hasta qué hora puedo ir?"*

Para responder hay que unir **dos datos de dos secciones**: dónde se paga en efectivo (facturación) y el horario de las agencias en sábado (contacto de agencias). Una sola búsqueda con la pregunta completa produce **un** vector que no se parece del todo a ninguna de las dos.

## Descomposición

Pedimos a un LLM que **separe la pregunta en subpreguntas** de un solo tema, buscamos **cada una** y fusionamos los resultados con **RRF** (*reciprocal rank fusion*: cada fragmento suma 1 ÷ (60 + posición) en cada lista donde aparece, así que no hace falta comparar puntajes entre búsquedas distintas).

```
pregunta ──► LLM ──► subpregunta 1 ──► buscar ─┐
                     subpregunta 2 ──► buscar ─┼──► fusión RRF ──► top-k ──► prompt
             pregunta original ───────► buscar ─┘
```

💰 **FYI:** esta técnica añade **1 llamada corta** al LLM por pregunta (≈ US$ 0.0002 con un modelo de proveedor; gratis con el local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0006 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    for id_ in ("MS01", "MS03", "G01"):
        p = next(x for x in GOLDEN if x["id"] == id_)
        subs, llamadas = descomponer(p["pregunta"])
        for l in llamadas: gasto.registrar(l)
        print(f"{id_}: {p['pregunta']}")
        for s in subs: print(f"     → {s}")
        print()
else:
    print("⚪ Celda de pago desactivada.")

Observa que una pregunta de **un solo tema** (`G01`) se descompone en lo mismo o en variantes: la técnica no estorba, pero cuesta una llamada. Ahora la medimos (solo recuperación) y la comparamos con la base.

## HyDE (*Hypothetical Document Embeddings*)

Una **pregunta** y su **respuesta** se parecen menos de lo que crees: "¿Hasta cuándo puedo pagar?" y "La factura vence el día 15…" casi no comparten palabras. HyDE pide al LLM que **escriba una respuesta hipotética** (aunque invente los datos) y **busca con el vector de ese texto**, que se parece más a los fragmentos del manual. El texto inventado jamás se muestra al usuario: solo sirve para buscar.

Es una de varias formas de *transformar la consulta* antes de buscar (otra: generar varias paráfrasis y fusionar, "multi-consulta"). La **reescritura de consulta** como tal la viste en la Sesión 6.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.01 con un modelo de proveedor: 2 técnicas × 24 preguntas × 1 llamada corta; gratis con el modelo local)
if PUEDE_GASTAR:
    t0 = time.perf_counter()
    desc = RecuperadorDescompuesto(base, k_sub=3)
    hyde = RecuperadorHyDE(base)
    informes_q = {
        "base k=4": inf_base_rec,
        "descomposición": evaluar_rag(None, GOLDEN, k=4, con_generacion=False, recuperador=desc),
        "HyDE": evaluar_rag(None, GOLDEN, k=4, con_generacion=False, recuperador=hyde),
    }
    print(tabla_comparativa(informes_q, ("recall", "precision", "mrr", "hit")))
    print(f"\n({time.perf_counter() - t0:.0f} s)\n")
    for nombre, inf in informes_q.items():
        pt = inf.por_tipo()
        print(f"{nombre:<16}" + "  ".join(f"{t}: {pt[t]['recall']:.0%}" for t in pt if pt[t]["n_con_respuesta"]))
else:
    print("⚪ Celda de pago desactivada.")

### Cómo leerlo

- La **descomposición** debería ayudar justo en `multi_salto` y `multi_seccion`, que son las preguntas para las que existe. Si en `directa` no cambia nada, está bien: no era su trabajo.
- **HyDE** es más **irregular**: cuando el modelo inventa una respuesta con el vocabulario correcto, ayuda; cuando inventa algo ajeno, te lleva a un fragmento equivocado. Con un modelo pequeño, suele empeorar la precisión y el MRR. Es una técnica para **probar y medir**, no para activar por defecto.
- Fíjate también en el costo en segundos: cada técnica con LLM añade latencia a **cada** pregunta.

> 🧰 **Regla práctica:** usa una técnica cuando puedas nombrar **la falla que ataca** y puedas **medir** que la arregla. Si no, es complejidad sin evidencia.

---
# Parte 5 · GraphRAG: qué es y cuándo conviene

## Un problema que los vectores no resuelven bien

Cambiamos de documento. Ahora es el **catálogo de servicios de TI** de Andes Fibra (ficticio), el tipo de cosa que vive en una **CMDB** (base de configuración de una mesa de ayuda): cada servicio, de qué depende y quién lo administra.

> *"Si falla la base de datos de clientes, ¿qué servicios dejan de funcionar?"*

El catálogo solo dice, en la sección de cada servicio, **de qué depende**. **Nadie escribió** "de la base de datos dependen X, Y y Z". La respuesta hay que **construirla recorriendo las dependencias al revés**, varios saltos. Un RAG vectorial busca *texto parecido a la pregunta*: encuentra la sección de la base de datos, que no contiene la respuesta.

## ¿Qué es GraphRAG?

**Un RAG que, además de fragmentos de texto, usa un grafo de conocimiento.**

- **Grafo de conocimiento:** una red de **entidades** (nodos: servicios, equipos, personas, productos) unidas por **relaciones** (aristas: "depende de", "administrado por"). Cada hecho es una **tripleta** `(sujeto, relación, objeto)`.
- **Pipeline típico:**

```
 documentos ──► (LLM) extrae entidades y relaciones ──► grafo de conocimiento
                                                              │
        ┌───────────── búsqueda LOCAL ─────────────┐          │
        │  entidades de la pregunta → sus vecinos  │◄─────────┤
        └──────────────────────────────────────────┘          │
        ┌───────────── búsqueda GLOBAL ────────────┐          │
        │  comunidades del grafo → resúmenes       │◄─────────┘
        └──────────────────────────────────────────┘
                              │
                     contexto ──► LLM ──► respuesta
```

## Implementaciones reales

| Proyecto | Qué es (según su documentación) |
| --- | --- |
| **[Microsoft GraphRAG](https://microsoft.github.io/graphrag/)** | Enfoque "estructurado y jerárquico" del RAG: divide el corpus en *text units*, **extrae entidades, relaciones y afirmaciones con un LLM**, agrupa el grafo en **comunidades** (algoritmo de Leiden) y **genera un resumen por comunidad**. Modos de consulta: **global** (preguntas sobre todo el corpus, con los resúmenes), **local** (entidades y sus vecinos), **DRIFT** (local con contexto de comunidad) y **básica** (búsqueda vectorial clásica). Su propia documentación recomienda *ajustar los prompts* antes de fiarse de los resultados. |
| **[LightRAG](https://github.com/HKUDS/LightRAG)** | Alternativa "ligera" (paper en EMNLP 2025, MIT): guarda **grafo y vectores a la vez**, un LLM extrae entidades y relaciones, y ofrece modos *naive*, *local*, *global*, *hybrid* y *mix* (el predeterminado). Permite **añadir documentos sin reindexar todo**, y evita los resúmenes por comunidad para gastar menos llamadas al LLM. |

> 🚫 **No los instalamos hoy.** Son paquetes pesados y su indexado hace **muchas** llamadas al LLM (cuesta dinero y tiempo). Hoy construimos a escala de juguete, con `networkx`, **lo que hay dentro**: tripletas, grafo, recorrido y contexto. Con eso entiendes qué hacen y qué te cuestan.

## ¿Cuándo aporta y cuándo NO?

| Aporta cuando… | No compensa cuando… |
| --- | --- |
| Las preguntas son **relacionales** ("¿qué se afecta si cae X?", "¿quién depende de quién?") | Las preguntas son **directas** y la respuesta está en un párrafo |
| Son **globales** ("¿cuál es el tema principal del corpus?", "¿cuál es el servicio más crítico?") | Tu corpus es pequeño y cabe en el contexto |
| Hay entidades con **muchas conexiones** (CMDB, contratos, redes de personas, cadena de suministro) | Tus documentos cambian a diario (**reindexar el grafo cuesta**) |
| Necesitas **explicar el camino** de la respuesta | No puedes validar la **calidad de la extracción** (un grafo con errores contamina todo) |

**Costos que casi nadie cuenta:** extraer el grafo = **1 o más llamadas al LLM por fragmento** del corpus (en un corpus grande, miles); mantenerlo al día; y revisar su calidad. Antes de adoptar GraphRAG, pregúntate si un RAG con buen *chunking*, *small-to-big* y descomposición ya resuelve tus preguntas.

In [ ]:
catalogo = leer_documento("datos/servicios_ti.md")
print(f"{len(catalogo.secciones)} secciones · ejemplo:\n")
for titulo, texto in catalogo.secciones[10:13]:
    print(f"## {titulo}\n{texto}\n")

Observa el patrón: cada sección dice **de qué depende** ese servicio, nunca **qué depende de él**. Esa información está *implícita*, repartida por todas las demás secciones.

### El grafo curado a mano

Para tener una base fiable, las tripletas del catálogo están escritas a mano en `datos/servicios_ti_grafo.json` (40 tripletas, 24 entidades). **Es el camino fiable**: si el grafo está mal, todo lo demás está mal. Más abajo veremos qué pasa cuando lo extrae un LLM.

In [ ]:
G = cargar_grafo()
print(f"{G.number_of_nodes()} entidades · {G.number_of_edges()} relaciones\n")
for a, b, d in list(G.edges(data=True))[:6]:
    print(f"  ({a}) ──{d['relacion']}──► ({b})")

dibujar_grafo(G, titulo="Servicios de TI de Andes Fibra · A → B significa 'A depende de B'")
plt.show()

### El grafo se consulta como una base de datos

Con el grafo, la pregunta de impacto **no necesita un LLM**: es un algoritmo (recorrer las aristas al revés).

In [ ]:
nodo = "Base de datos de clientes"
afectados = afectados_por(G, nodo)
print(f"Si falla «{nodo}», dejan de funcionar {len(afectados)} servicios:")
for s in afectados: print("   •", s)

print("\n¿De qué depende el Chat de soporte, directa o indirectamente?")
print("   ", ", ".join(dependencias_de(G, "Chat de soporte")))

print("\nServicios con más servicios que dependen DIRECTAMENTE de ellos (pregunta 'global': hay que contar):")
for n, c in mas_dependidos(G): print(f"   {c} · {n}")

dibujar_grafo(G, resaltar=[nodo, *afectados], titulo=f"Si falla «{nodo}» (rojo = origen y afectados)")
plt.show()

Eso es lo que **un grafo da y un vector no**: *recorrer*, *contar*, *agregar*. Ahora veamos qué hace la búsqueda vectorial con la misma pregunta.

In [ ]:
vec = crear_almacen("memoria")
ingestar(vec, "datos/servicios_ti.md", estrategia="oraciones", max_palabras=70, solape=1)

pregunta = "Si falla la base de datos de clientes, ¿qué servicios dejan de funcionar?"
print(pregunta, "\n\nTop-4 vectorial:\n")
for i, r in enumerate(recuperar(vec, pregunta, k=4), start=1):
    print(f"  {i}. {r.puntaje:.3f}  {r.metadatos['seccion']}")
print("\nDebería traer (porque dependen de ella, directa o indirectamente):", ", ".join(afectados))

El vectorial trae los fragmentos **sobre** la base de datos y algunos servicios que la *nombran*, pero deja fuera varios afectados **indirectos** (el chat de soporte depende de tickets, que depende de la base de datos: dos saltos). Es exactamente el tipo de pregunta para el que se inventó GraphRAG.

---
# Parte 6 · GraphRAG medido: vectorial vs grafo vs combinado

Hay 8 preguntas **multi-salto** en `datos/preguntas_multisalto.json`, con la lista exacta de entidades que debe contener la respuesta (calculada con el grafo curado). Medimos **dos cosas por separado**, como siempre:

| Capa | Métrica | Pregunta |
| --- | --- | --- |
| Recuperación | **cobertura del contexto** | ¿Las entidades de la respuesta llegaron al prompt? |
| Generación | **recall y precisión de la respuesta** | ¿El modelo nombró todas y solo esas? |

## Los recuperadores que comparamos

| Recuperador | Qué entrega al modelo |
| --- | --- |
| `vectorial` (k = 4 y k = 8) | Los fragmentos del catálogo más parecidos a la pregunta |
| `grafo · vecindad` | Los **hechos** a pocos saltos de las entidades de la pregunta (*búsqueda local*) |
| `grafo · recorrido` | Esos hechos **más un análisis calculado** con un algoritmo del grafo (impacto, dependencias, ranking) |
| `vectorial + recorrido` | Lo de los dos mundos juntos |

Primero, **solo recuperación** (gratis): ¿llega la información al prompt?

In [ ]:
PREGUNTAS_MH = json.loads(Path("datos/preguntas_multisalto.json").read_text(encoding="utf-8"))
rv = lambda pregunta, k: recuperar(vec, pregunta, k=k)

RECUPERADORES = {
    "vectorial k=4": (rv, 4),
    "vectorial k=8": (rv, 8),
    "grafo vecindad (1 salto)": (RecuperadorGrafo(G, profundidad=1), 4),
    "grafo vecindad (2 saltos)": (RecuperadorGrafo(G, profundidad=2), 4),
    "grafo vecindad (3 saltos)": (RecuperadorGrafo(G, profundidad=3), 4),
    "grafo recorrido": (RecuperadorGrafo(G, profundidad=2, modo="recorrido"), 4),
    "vectorial + recorrido": (RecuperadorCombinado(rv, RecuperadorGrafo(G, profundidad=2, modo="recorrido")), 4),
}
print(f"{'recuperador':<28}{'cobertura':>10}{'palabras':>10}   por pregunta (MH1..MH8)")
for nombre, (rec, k) in RECUPERADORES.items():
    filas = evaluar_multisalto(G, PREGUNTAS_MH, rec, k=k, con_generacion=False)
    s = resumen_multisalto(filas)
    print(f"{nombre:<28}{s['contexto_recall']:>10.0%}{s['palabras_contexto']:>10.0f}   " + " ".join(f"{f['contexto_recall']:.1f}" for f in filas))

### Lo que se ve

- **Vectorial**: subir `k` mejora la cobertura, pero a costa de **más palabras** de contexto, y aun así deja agujeros.
- **Grafo por vecindad**: cada salto de profundidad trae más hechos; con 3 saltos casi todo llega... pero también llega más ruido. La profundidad es un *k* disfrazado.
- **Recorrido**: el algoritmo entrega **exactamente** lo que se pidió, con poco texto.
- **Combinado**: cubre también las preguntas donde la entidad no aparece en el texto de la pregunta (la "global").

Pero *recuperar bien* no es *responder bien*. Ahora la **generación** (💰 una llamada por pregunta, gratis con el modelo local):

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.002 por recuperador con un modelo de proveedor: 8 llamadas cada uno; gratis con el modelo local)
if PUEDE_GASTAR:
    resultados_mh = {}
    for nombre in ("vectorial k=4", "vectorial k=8", "grafo vecindad (3 saltos)", "grafo recorrido", "vectorial + recorrido"):
        rec, k = RECUPERADORES[nombre]
        resultados_mh[nombre] = evaluar_multisalto(G, PREGUNTAS_MH, rec, k=k, version="v2")
    print(f"{'recuperador':<28}{'ctx':>6}{'resp. recall':>14}{'resp. precisión':>17}{'completas':>11}")
    for nombre, filas in resultados_mh.items():
        s = resumen_multisalto(filas)
        print(f"{nombre:<28}{s['contexto_recall']:>6.0%}{s['respuesta_recall']:>14.0%}{s['respuesta_precision']:>17.0%}{s['completas']:>8}/{s['n']}")
else:
    print("⚪ Celda de pago desactivada.")

Compara las columnas **ctx** (¿llegó?) y **resp. recall** (¿lo usó?). Si en una fila la primera es alta y la segunda baja, **el problema ya no es la recuperación**: el modelo tiene los hechos pero no sabe encadenarlos. Un modelo pequeño con una lista de hechos de dos o tres saltos suele fallar; por eso el **recorrido** (que le entrega el resultado ya calculado) rinde mucho más.

Veamos respuestas concretas:

In [ ]:
if PUEDE_GASTAR:
    for id_ in ("MH1", "MH4", "MH7"):
        p = next(x for x in PREGUNTAS_MH if x["id"] == id_)
        print(f"{id_} · {p['pregunta']}")
        print(f"   esperado: {', '.join(p['esperadas'])}")
        for nombre in ("vectorial k=4", "grafo recorrido"):
            fila = next(f for f in resultados_mh[nombre] if f["id"] == id_)
            print(f"   {nombre:<16}: {fila['respuesta'][:170]!r}  (faltan {fila['faltan']})")
        print()

**`MH4`** ("¿qué equipo administra el servicio donde guarda sus datos la API de facturación?") suele fallar con **todos** los recuperadores: el hecho está en el contexto, pero el modelo contesta con el equipo de la *API*, no el de la *base de datos*. Es una falla de **razonamiento encadenado** (generación), no de recuperación. Ningún recuperador la arregla; la arreglan un modelo más fuerte, un prompt que obligue a razonar paso a paso o... un agente que consulte el grafo en pasos (lo veremos).

---
## El paso frágil: extraer el grafo con un LLM

Hasta aquí usamos un grafo **curado a mano**. En la vida real nadie escribe 40 tripletas por documento: **un LLM las extrae del texto**. Es, de lejos, la parte más cara y más frágil de GraphRAG, y la que casi nunca se mide. Midámosla contra el grafo curado, con dos versiones del prompt:

- **v1**: nombres libres.
- **v2**: se le da la **lista de entidades conocidas** y debe usar solo esas.

💰 **FYI:** 1 llamada por sección (16 secciones por versión); gratis con el modelo local, ≈ US$ 0.003 con un modelo de proveedor.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.006 con un modelo de proveedor: 2 versiones × 16 secciones; gratis con el modelo local)
if PUEDE_GASTAR:
    extraccion = {}
    for version in ("v1", "v2"):
        t0 = time.perf_counter()
        tripletas, llamadas = extraer_tripletas("datos/servicios_ti.md", version=version, entidades=list(G.nodes))
        for l in llamadas: gasto.registrar(l)
        c = comparar_tripletas(G, tripletas)
        extraccion[version] = (tripletas, c)
        print(f"Prompt {version} · {time.perf_counter() - t0:.0f} s · {c['extraidas']} tripletas extraídas "
              f"(el grafo curado tiene {c['curadas']})")
        print(f"   correctas {c['correctas']} · precisión {c['precision']:.0%} · recall {c['recall']:.0%} · F1 {c['f1']:.0%}")
        print(f"   con una entidad que no existe en el catálogo: {c['con_entidad_no_reconocida']}  {c['nombres_no_reconocidos'][:4]}")
        print(f"   tripletas inventadas o mal formadas (muestra): {c['sobran'][:3]}\n")
else:
    print("⚪ Celda de pago desactivada.")

### Lo que enseña esta medición

1. **Con nombres libres (v1)** el modelo escribe "Directorio de usuarios y Base de datos de clientes" como *una* entidad, o "centros de datos", y esas aristas **no se pueden unir** al resto del grafo.
2. **Darle el vocabulario (v2)** mejora mucho el *recall* (encuentra casi todo), pero **la precisión sigue lejos del 100 %**: inventa relaciones que el texto no dice (aristas de más) o invierte el sentido.
3. **Una arista falsa no es un error aislado**: cambia la respuesta de **todos los recorridos** que pasan por ella. Veámoslo.

In [ ]:
if PUEDE_GASTAR:
    G_ext = grafo_desde_tripletas(G, extraccion["v2"][0])
    print(f"Grafo extraído (v2): {G_ext.number_of_edges()} relaciones (el curado tiene {G.number_of_edges()})\n")
    print(f"{'pregunta':<6}{'curado':>8}{'extraído':>10}   ¿qué cambia?")
    for p in PREGUNTAS_MH:
        if p["tipo"] != "impacto":
            continue
        nodo = entidades_en(G, p["pregunta"])[0]
        bien, mal = set(afectados_por(G, nodo)), set(afectados_por(G_ext, nodo))
        sobran, faltan = sorted(mal - bien), sorted(bien - mal)
        print(f"{p['id']:<6}{len(bien):>8}{len(mal):>10}   sobran {len(sobran)}" + (f" ({', '.join(sobran[:3])}…)" if sobran else "")
              + f" · faltan {len(faltan)}")

**Moraleja:** el grafo es tan bueno como su extracción, y **hay que medir la extracción** (precisión y recall de tripletas contra una muestra curada) **antes** de confiar en las respuestas. Es el mismo principio de hoy aplicado a otra capa: *no se confía en lo que no se mide*.

---
## RAG agéntico (un adelanto)

Hasta aquí, **tú** decidiste qué técnica usar para cada pregunta. En un **RAG agéntico**, un **agente** (un LLM con herramientas) decide **en cada paso**: *¿busco en el manual? ¿consulto el grafo? ¿descompongo la pregunta? ¿ya tengo suficiente para responder o busco otra vez?* Itera hasta estar satisfecho.

| | RAG (lo de hoy) | RAG agéntico |
| --- | --- | --- |
| Flujo | Fijo: buscar → armar prompt → responder | Dinámico: el modelo decide los pasos |
| Búsquedas por pregunta | 1 (o las que tú programaste) | Las que el agente necesite |
| Costo y latencia | Predecibles | Variables (a veces mucho mayores) |
| Evaluación | Golden set y métricas de hoy | Las mismas **más** evaluar las decisiones del agente |
| Riesgo | Contexto no confiable | Lo anterior **más** acciones: mínimo privilegio |

No lo construimos hoy: **primero hay que dominar lo que el agente va a orquestar**. Cuando llegues a agentes, el recuperador vectorial, el de grafo y la descomposición serán *herramientas* que el agente invoca, y el golden set de hoy será tu prueba de regresión.

## 🧭 Qué técnica para qué falla

| Falla que mides | Técnica | Costo extra |
| --- | --- | --- |
| Falta contexto alrededor del fragmento correcto | *Small-to-big* (sección o ventana) | Más tokens por pregunta |
| Fragmentos genéricos o ambiguos | Encabezados contextuales | Nada |
| La pregunta necesita **dos temas** | Descomposición + RRF | 1 llamada al LLM por pregunta |
| La pregunta no se parece al texto de la respuesta | HyDE (¡mídelo!) | 1 llamada al LLM por pregunta |
| Falta el vocabulario exacto (códigos, nombres) | Búsqueda híbrida (Sesión 6) | Un índice más |
| El orden del top-k es malo | Reranking (Sesión 6) | Un modelo más por pregunta |
| Preguntas **relacionales o globales** | **GraphRAG** | Extraer, validar y mantener el grafo |
| La respuesta requiere varios pasos y decisiones | RAG agéntico | Latencia, costo y evaluación mayores |

---
# Parte 7 · Evaluar la generación y diagnosticar por capa

Volvemos al manual de Andes Fibra y al golden set de 24 preguntas. Hasta ahora medimos solo la **recuperación**. Ahora medimos la respuesta del modelo con **comprobaciones por código** (baratas, deterministas, repetibles):

| Comprobación | Qué detecta | Cómo |
| --- | --- | --- |
| **Dato correcto** | La respuesta trae el dato clave esperado | `claves` del golden set aparecen en el texto |
| **Citas válidas** | El modelo cita fragmentos que existen | `[n]` dentro de los fragmentos entregados |
| **"No sé" bien dicho** | Se abstiene en las preguntas fuera de alcance | La frase exacta del prompt |
| **Cifras con respaldo** | Una cifra de la respuesta que no está en el contexto (un *checksum* de la alucinación) | Números de la respuesta ⊆ números del contexto ∪ pregunta |

### El diagnóstico por capa

La columna más útil de un informe: **¿en qué capa falló cada pregunta?** Para cada una, la primera categoría que aplique:

| Diagnóstico | Capa | Significa |
| --- | --- | --- |
| `abstencion_por_recuperacion` / `recuperacion_incompleta` | **recuperación** | La información necesaria no llegó al prompt; el modelo hizo lo que pudo |
| `abstencion_indebida` | generación | Llegó todo, pero el modelo dijo "no sé" |
| `dato_incorrecto` | generación | Llegó todo, pero el dato no está en la respuesta |
| `cifra_sin_respaldo` / `alucinacion` | generación (**alucinación**) | Inventó una cifra, o contestó algo que no se podía contestar |
| `sin_citas_validas` | verificabilidad | Respondió bien, pero sin citar o citando mal |

💰 **FYI:** 24 llamadas al LLM (≈ US$ 0.012 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.012 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    inf_v1 = correr("v1 · k=4", base, k=4, version="v1")
    imprimir_resumen(inf_v1, "RAG de la Sesión 5")
    print("\n" + tabla_por_tipo(inf_v1))
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    print("Dónde falló cada pregunta:\n")
    for diag, ids in inf_v1.diagnosticos().items():
        print(f"{diag:<30} capa: {CAPA_DEL_DIAGNOSTICO[diag]:<26} {', '.join(ids)}")

    print("\nUn ejemplo de cada diagnóstico:\n")
    mostrados = set()
    for f in inf_v1.filas:
        if f.diagnostico not in ("ok", *mostrados):
            mostrados.add(f.diagnostico)
            print(f"[{f.id} · {f.diagnostico}] {f.pregunta}\n     → {f.respuesta[:210]!r}\n")

### Cómo leer este informe

1. **Mira la capa antes que el porcentaje.** Si los fallos son de *recuperación*, cambia el recuperador (Partes 3 a 6); si son de *generación*, cambia el prompt o el modelo. Tocar la capa equivocada no mueve nada.
2. **"0 alucinaciones" puede esconder otro problema.** Un modelo que dice "no sé" a todo no alucina nunca... y no sirve. Mira siempre **juntas** las dos columnas: alucinaciones y *"no sé" indebido*. Bajar una a costa de subir la otra no es mejorar.
3. **Las comprobaciones por código se equivocan, y hay que conocer cómo.** Una respuesta como *"No, no es normal."* (para el fraude del SMS) es correcta pero no contiene la palabra clave `fraude`: la marcamos mal (**falso negativo**). Lo contrario también pasa: la cifra de un concepto equivocado (el 15 del vencimiento usado como día de suspensión) **sí está en el contexto**, así que el chequeo de cifras no la ve. Las comprobaciones por código son el piso, no el techo: por eso existe el juez de la Parte 9.
4. **Con un modelo local de 3B parámetros** verás más abstenciones indebidas y citas flojas que con un modelo de proveedor. No es un defecto de tu código: es el modelo. Cambia `RAG_PROVIDER` y compara.

> ⚙️ **Ruido de ejecución.** Aun con temperatura 0, un servidor local puede dar respuestas ligeramente distintas entre corridas. Si una pregunta cambia de diagnóstico al repetir, es ruido: por eso comparamos con **muchas preguntas** y miramos diferencias grandes.

---
# Parte 8 · Alucinaciones: causas, mitigación y regresión

Una **alucinación** es una respuesta plausible pero no respaldada (o falsa). En un RAG casi nunca es "el modelo se volvió loco": tiene **causas identificables**, y cada una tiene su mitigación.

| Causa | Cómo se ve | Mitigación |
| --- | --- | --- |
| **Recuperación pobre** | El fragmento correcto no llegó; el modelo rellena | Mejorar recuperación; **abstenerse** |
| **El prompt no permite el "no sé"** | Contesta siempre, aunque no sepa | Frase exacta de abstención; probar con preguntas fuera de alcance |
| **Pregunta "parecida pero ausente"** | El tema suena al manual, el dato no está | "No sé" explícito para temas parecidos (prompt v3); evaluarlo |
| **Contexto contradictorio u obsoleto** | El modelo **cita fielmente** un dato viejo | **Higiene de la base**: versiones, vigencia, metadatos y filtros |
| **El modelo ignora el contexto** | Usa su "memoria" y no lo recuperado | Modelo más capaz; verificar con juez y con cifras |
| **Extrapolar o deducir** | "Como el de 600 cuesta 55, el de 1 Gbps costaría ~90" | Prohibir estimar; verificar cifras |

## Demo 1 · Un prompt sin salida de escape
Dos prompts con las mismas preguntas "trampa" (fuera de alcance pero parecidas al manual): uno **ingenuo** (sin regla de "no sé") y el v3 del curso.

💰 **FYI:** 4 llamadas (≈ US$ 0.002 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.002 con un modelo de proveedor; gratis con el modelo local)
def prompt_ingenuo(pregunta, fragmentos):
    # Sin regla de "no sé", sin citas, sin 'solo el contexto': el prompt de quien no ha medido nunca.
    ctx = "\n\n".join(f"[{i}] {f.texto}" for i, f in enumerate(fragmentos, start=1))
    return f"Responde la pregunta del cliente con este contexto.\n\n{ctx}\n\nPregunta: {pregunta}"

if PUEDE_GASTAR:
    for id_ in ("F02", "F01"):
        p = next(x for x in GOLDEN if x["id"] == id_)
        frs = recuperar(base, p["pregunta"], k=4)
        for etiqueta, prompt in (("ingenuo", prompt_ingenuo(p["pregunta"], frs)), ("v3", construir_prompt(p["pregunta"], frs, "v3"))):
            r = gasto.registrar(generar(prompt, max_tokens=300))
            print(f"[{id_} · {etiqueta}] {p['pregunta']}\n     → {r.texto[:260]!r}\n")
else:
    print("⚪ Celda de pago desactivada.")

Con el prompt ingenuo, el modelo suele **disculparse y luego deducir** ("podemos inferir que… es posible que…"): una alucinación con buenos modales. Con el v3, responde la frase exacta. **La diferencia entre las dos es una línea del prompt, y solo la ves si evalúas con preguntas fuera de alcance.**

## Demo 2 · Contexto obsoleto: fiel, pero incorrecto

Alguien dejó en la base una **tabla de precios anterior** (35 dólares en vez de 39). Se indexa junto al manual. ¿Qué pasa?

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 con un modelo de proveedor; gratis con el modelo local)
VIEJO = Fragmento(
    id="tabla_vieja#0", metadatos={"fuente": "precios_2025.md", "seccion": "1. Planes de internet", "n": 0, "vigencia": "anterior"},
    texto="Tabla de precios anterior (vigente hasta el año pasado): el plan Familiar de 300 Mbps cuesta 35 dólares al mes.",
)
VIEJO.vector = vectorizar([texto_a_vectorizar(VIEJO)])[0]

contaminada = crear_almacen("memoria")
ingestar_con_metadatos(contaminada, MANUAL, {"vigencia": "actual"})
contaminada.insertar([VIEJO])

p = next(x for x in GOLDEN if x["id"] == "G01")
frs = recuperar(contaminada, p["pregunta"], k=4)
print("Top-4:", [(f.id, round(f.puntaje, 2)) for f in frs], "\n")

if PUEDE_GASTAR:
    r = gasto.registrar(generar(construir_prompt(p["pregunta"], frs, "v3"), max_tokens=300))
    print("Respuesta:", r.texto)
    print("\n¿Contiene el dato correcto (39)?       ", contiene_claves(r.texto, p["claves"]))
    print("Cifras sin respaldo en el contexto:     ", numeros_sin_respaldo(r.texto, frs, p["pregunta"]) or "ninguna  ← el chequeo de cifras NO la ve")

La respuesta **cita fielmente** un fragmento que está en el contexto: el juez de fidelidad la aprobaría y el chequeo de cifras también. Pero **es incorrecta**. Esto separa dos ideas que se confunden:

| | Pregunta | Se mide con |
| --- | --- | --- |
| **Fidelidad** (*groundedness*) | ¿Lo que dice está **en el contexto**? | Cifras con respaldo, juez de fidelidad |
| **Corrección** | ¿Lo que dice es **verdad**? | Respuesta esperada del golden set |

Un RAG puede ser **fiel e incorrecto** (contexto malo) o, más raro, **correcto e infiel** (acertó de memoria). Hay que medir las dos.

**La mitigación no está en el prompt, está en los datos:** guardar un metadato de **vigencia** y **filtrar** al buscar.

In [ ]:
filtrado = lambda pregunta, k: contaminada.buscar(vector_consulta(pregunta), k=k, filtro={"vigencia": "actual"})
frs_ok = filtrado(p["pregunta"], 4)
print("Top-4 con filtro de vigencia:", [(f.id, round(f.puntaje, 2)) for f in frs_ok])
if PUEDE_GASTAR:
    r = gasto.registrar(generar(construir_prompt(p["pregunta"], frs_ok, "v3"), max_tokens=300))
    print("Respuesta:", r.texto, "\n¿Contiene el dato correcto?", contiene_claves(r.texto, p["claves"]))

Es el mismo principio que **retirar una imagen de contenedor obsoleta del registro**: da igual lo bien que esté escrito el *deployment*, si el registro sirve la imagen vieja. (Y por eso el filtro por metadatos, que también sirve para permisos, se aplica **en la recuperación**, nunca confiando en el prompt.)

## Medir → cambiar → volver a medir

Ahora el ciclo completo, con el golden set de 24 preguntas y la generación. Cada fila cambia **una cosa** respecto a la anterior:

| Configuración | Qué cambia |
| --- | --- |
| `v1 · k=4` | El RAG de la Sesión 5 (ya medido) |
| `v2 · k=4` | Prompt v2: copiar cifras tal cual y respuestas cortas |
| `v3 · k=4` | Prompt v3: prohíbe estimar, extiende el "no sé" a lo parecido-ausente, cubre las interpretaciones de una pregunta ambigua |
| `s2b sección k=2 · v3` | Recuperación **small-to-big** (Parte 3) con el prompt v3 |
| `descomposición · v3` | Recuperación con **descomposición** (Parte 4) con el prompt v3 |

💰 **FYI:** 4 configuraciones nuevas × 24 preguntas = 96 llamadas (≈ US$ 0.05 con un modelo de proveedor, más ≈ 24 llamadas cortas de la descomposición; gratis con el modelo local). Tarda unos minutos con el local.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.05 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    t0 = time.perf_counter()
    s2b = RecuperadorSmallToBig(peq, MANUAL, frs_peq, modo="seccion")
    correr("v2 · k=4", base, k=4, version="v2")
    correr("v3 · k=4", base, k=4, version="v3")
    correr("s2b sección k=2 · v3", None, k=2, version="v3", recuperador=s2b)
    correr("descomposición · v3", None, k=4, version="v3", recuperador=RecuperadorDescompuesto(base, k_sub=3))
    ciclo = {n: INFORMES[n] for n in ("v1 · k=4", "v2 · k=4", "v3 · k=4", "s2b sección k=2 · v3", "descomposición · v3")}
    print(f"({time.perf_counter() - t0:.0f} s)\n")
    print(tabla_comparativa(ciclo, ("recall", "precision", "datos_correctos", "citas_validas", "abstencion_correcta",
                                    "abstencion_indebida", "alucinaciones", "ok")))
else:
    print("⚪ Celda de pago desactivada.")

### Cómo leer la tabla

- Mira **juntas** *datos correctos*, *"no sé" bien dicho*, *"no sé" indebido* y *alucinaciones*. Una mejora real sube las dos primeras sin disparar las otras.
- Cambiar el **prompt** mueve la **generación** (datos, citas, abstenciones) y deja intactos Recall y Precision. Cambiar el **recuperador** mueve Recall/Precision y, a través del contexto, la generación. **Cada cambio toca una capa**: por eso se mide por capas.
- Una diferencia de 1 pregunta (≈ 4 puntos) es ruido; una de 4 o 5 es señal.
- Puede ocurrir que **ninguna configuración sea la mejor en todo**. Eso es lo normal: elegir es decidir qué falla te cuesta más (¿una respuesta inventada o un "no sé" de más?).

## El SLO: ¿puedo desplegar este cambio?

Un SLO convierte el informe en una **decisión**. Los valores por defecto (en `evaluacion.SLO_POR_DEFECTO`) son solo un ejemplo: en un proyecto real los fija quien responde por el servicio.

In [ ]:
print("SLO:", {k: f"{op} {v:.0%}" for k, (op, v) in SLO_POR_DEFECTO.items()}, "\n")
for nombre, inf in INFORMES.items():
    filas = verificar_slo(inf.resumen())
    estado = "✅ APRUEBA" if all(f["cumple"] for f in filas) else "❌ NO APRUEBA"
    detalle = " · ".join(f"{f['metrica']} {f['valor']:.0%} {'✓' if f['cumple'] else '✗'}" for f in filas)
    print(f"{estado:<14}{nombre:<26}{detalle}")

## Mitigación 3 · El umbral de similitud (y por qué no basta)

Un **umbral** responde "no sé" sin siquiera llamar al LLM cuando el mejor fragmento no se parece a la pregunta. Ahorra dinero, pero ¿separa las preguntas con respuesta de las que no? Veamos los puntajes del mejor fragmento (gratis):

In [ ]:
pm = puntajes_mejor_fragmento(base, GOLDEN)
for grupo, lista in pm.items():
    print(f"{grupo:<18}", " ".join(f"{s:.2f}" for _, s in sorted(lista, key=lambda x: x[1])))
mejor_ajena = max(s for i, s in pm["fuera_de_alcance"] if i == "F04")
peor_con = min(s for _, s in pm["con_respuesta"])
print(f"\nLa pregunta AJENA saca {mejor_ajena:.2f}: un umbral la descarta.")
print(f"Las preguntas con respuesta llegan a sacar {peor_con:.2f}; las fuera-de-alcance PARECIDAS (F01 a F03) sacan puntajes dentro del mismo rango.")

El umbral descarta lo **ajeno**, pero las preguntas **parecidas pero ausentes** (televisión por cable, el plan de 1 Gbps) puntúan igual que las legítimas: ningún umbral las separa sin perder preguntas buenas. Para esas hace falta la **instrucción de "no sé" en el prompt (v3)** y **medirla**. Mitigar en capas, otra vez.

---
# Parte 9 · El juez LLM (*LLM-as-judge*)

Las comprobaciones por código no entienden el lenguaje: no saben que "no es normal" equivale a "es un fraude". Para eso se usa **otro modelo como juez**: le das una **rúbrica**, la pregunta, la respuesta y el contexto (o la respuesta esperada), y devuelve un **veredicto estructurado**.

| Juez | Pregunta | Recibe |
| --- | --- | --- |
| **Fidelidad** (*groundedness*) | ¿Todo lo que afirma la respuesta está en el contexto? | Contexto + respuesta |
| **Corrección** | ¿Coincide con la respuesta esperada? | Respuesta esperada + respuesta |

Buenas prácticas que ya conoces de la Sesión 3: **un criterio por juez**, rúbrica explícita, **salida estructurada validada** (Pydantic) y prompt **versionado**.

## Pero un juez es un instrumento... y hay que calibrarlo

Un medidor sin certificar no es un medidor. Los jueces LLM tienen **sesgos conocidos**:

| Sesgo | Qué hace |
| --- | --- |
| **Indulgencia** | Aprueba casi todo (el error más caro: deja pasar las alucinaciones) |
| **Severidad** | Rechaza de más (te hace perder tiempo revisando respuestas buenas) |
| **Sesgo de longitud** | Prefiere respuestas largas y detalladas, aunque un dato esté mal |
| **Autopreferencia** | Favorece respuestas escritas por un modelo parecido a él (por eso el juez ideal es **otro** modelo) |
| **Sensibilidad al prompt** | Cambia el veredicto al cambiar una frase de la rúbrica |

**La calibración** consiste en pasar al juez respuestas que **una persona ya etiquetó** y medir cuánto coinciden. Escribimos 21 respuestas con su etiqueta humana en `datos/etiquetas_juez.json`, incluidas trampas: una cifra equivocada que sí está en el contexto, una extrapolación disfrazada de prudencia, una respuesta larga con un error escondido.

In [ ]:
ETIQUETAS = json.loads(Path("datos/etiquetas_juez.json").read_text(encoding="utf-8"))
por_id = {p["id"]: p for p in GOLDEN}
for e in [x for x in ETIQUETAS if x["id"] in ("J02", "J07", "J14", "J16")]:
    print(f"{e['id']} · {por_id[e['pregunta_id']]['pregunta']}")
    print(f"   respuesta: {e['respuesta']}")
    print(f"   etiqueta humana → fiel: {e['fiel']} · correcta: {e['correcta']}  ({e['nota']})\n")
print(f"{len(ETIQUETAS)} etiquetas · {sum(e['fiel'] for e in ETIQUETAS)} fieles / {sum(not e['fiel'] for e in ETIQUETAS)} no fieles")

### Calibrar el juez

Medimos tres jueces con el mismo modelo, que es el que tengas configurado (`JUEZ_PROVIDER`/`JUEZ_MODEL`): el de **fidelidad v1** (sin rúbrica), el de **fidelidad v2** (rúbrica estricta) y el de **corrección v1**. Las medidas:

- **Exactitud:** fracción de veredictos que coinciden con la persona.
- **Kappa de Cohen:** acuerdo **descontando el azar** (1 = perfecto, 0 = tanto como lanzar una moneda). La exactitud sola engaña cuando las clases están desbalanceadas.
- **Indulgencia:** de lo que la persona rechazó, cuánto aprobó el juez. **Severidad:** de lo que la persona aprobó, cuánto rechazó el juez.

💰 **FYI:** 21 llamadas por juez (63 en total; ≈ US$ 0.03 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.03 con un modelo de proveedor; gratis con el modelo local)
def mostrar_calibracion(nombre, c):
    print(f"{nombre:<20} exactitud {c['exactitud']:.0%} · kappa {c['kappa']:.2f} · "
          f"indulgencia {c['indulgencia']:.0%} · severidad {c['severidad']:.0%} · sin veredicto {c['sin_veredicto']} · {c['segundos']:.0f} s")
    print(f"{'':<20} el juez aprobó bien {c['aprobo_bien']} · aprobó MAL {c['aprobo_mal']} · rechazó MAL {c['rechazo_mal']} · rechazó bien {c['rechazo_bien']}")

if PUEDE_GASTAR:
    calibraciones = {}
    for nombre, criterio, version in (("fidelidad v1", "fidelidad", "v1"), ("fidelidad v2", "fidelidad", "v2"), ("corrección v1", "correccion", "v1")):
        calibraciones[nombre] = calibrar_juez(ETIQUETAS, GOLDEN, MANUAL, criterio, version)
        mostrar_calibracion(nombre, calibraciones[nombre])
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    c = calibraciones["fidelidad v2"]
    print("Casos donde el juez de fidelidad v2 NO coincidió con la persona:\n")
    for f in c["filas"]:
        if f["humano"] != f["juez"]:
            print(f"  {f['id']}: persona={'fiel' if f['humano'] else 'no fiel'} · juez={'fiel' if f['juez'] else 'no fiel'}  ← {f['nota']}")

### Lo que enseña esta calibración

1. **El prompt del juez importa tanto como el del RAG.** Pasar de v1 (sin rúbrica) a v2 (rúbrica estricta) cambia el *tipo* de error: de un juez que **rechaza de más** a uno que deja pasar alucinaciones. Ninguno es bueno si el kappa es bajo.
2. **Un modelo pequeño es un mal juez.** Con un juez de 3B parámetros el acuerdo con las personas es **poco mejor que el azar**. Aprueba cifras equivocadas y extrapolaciones, y se confunde con las respuestas largas. Si tu juez es débil, **tu evaluación es decoración**.
3. **Qué hacer entonces:**
   - Usa un modelo **más capaz** que el generador y, si puedes, **distinto** (evita la autopreferencia).
   - **Calibra siempre** contra etiquetas humanas; fija una meta (p. ej. kappa ≥ 0.6) antes de confiar.
   - Usa el juez para **priorizar** qué revisa una persona, no para reemplazarla.
   - **Combina** el juez con las comprobaciones por código: se compensan.
   - Mantén el golden set y las etiquetas **versionados**: son tus pruebas de regresión.

In [ ]:
# Opcional · ¿Cambia el cuadro con un juez más grande?  Un modelo local más capaz, o un proveedor.
# 💰 FYI: con un modelo de proveedor ≈ US$ 0.01 (21 llamadas); con un modelo local grande tarda varios minutos.
MODELO_JUEZ_GRANDE = None      # p. ej. "google/gemma-4-12b-qat" (local) · déjalo en None para omitir esta celda
if PUEDE_GASTAR and MODELO_JUEZ_GRANDE:
    c_grande = calibrar_juez(ETIQUETAS, GOLDEN, MANUAL, "fidelidad", "v2", modelo=MODELO_JUEZ_GRANDE)
    mostrar_calibracion(f"v2 · {MODELO_JUEZ_GRANDE}", c_grande)

> 🔬 **Una referencia.** Con `google/gemma-4-12b-qat` (modelo local de 12B parámetros) el juez de fidelidad v2 coincidió con la persona en **21 de 21** casos (kappa 1.0; en 2 casos no devolvió un JSON válido), con más de un minuto por veredicto en un equipo sin GPU. El mismo prompt de rúbrica, con un modelo más capaz, pasó de azar a acuerdo casi total: **el modelo del juez pesa más que su prompt**. Pero 21 etiquetas son pocas: no lo tomes como garantía; una calibración seria necesita más casos y volver a hacerse al cambiar de modelo o de rúbrica.

### El juez y el chequeo por código, sobre respuestas reales del RAG

Cruzamos las dos herramientas sobre el subconjunto rápido (10 preguntas) con el prompt v3: ¿coinciden el juez de fidelidad y el chequeo de cifras con el diagnóstico?

💰 **FYI:** 10 llamadas del RAG + hasta 10 del juez (≈ US$ 0.01 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.01 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    rapido = [p for p in GOLDEN if p["rapido"]]
    inf_juez = evaluar_rag(base, rapido, k=4, version="v3", juez="fidelidad")
    print(f"{'id':<6}{'diagnóstico':<28}{'cifras sin respaldo':<22}{'juez de fidelidad'}")
    for f in inf_juez.filas:
        print(f"{f.id:<6}{f.diagnostico:<28}{str(f.numeros_sin_respaldo or '—'):<22}{'fiel' if f.fiel else ('NO fiel' if f.fiel is False else 'sin veredicto')}")
    print(f"\nFidelidad según el juez: {inf_juez.resumen().get('fidelidad_juez', 0):.0%}")

---
# Parte 10 · Práctica integrada: el Medidor de calidad RAG

La práctica de la sesión es [`practica/medidor_calidad_rag.py`](practica/medidor_calidad_rag.py): un programa único y comentado que **evalúa tu RAG con el golden set** y entrega un **informe** (consola, JSON y Markdown) con Recall@K, Precision@K, MRR, las métricas de generación, el diagnóstico por capa y si **cumple el SLO**. Si no lo cumple, termina con código de salida 1: sirve como **puerta de calidad en CI**.

```bash
# Desde la carpeta s07-rag-avanzado/
uv run python practica/medidor_calidad_rag.py --sin-generacion                       # solo recuperación (gratis, segundos)
uv run python practica/medidor_calidad_rag.py --rapido --version v3                  # las 10 preguntas rápidas
uv run python practica/medidor_calidad_rag.py --tecnica small-to-big -k 2 --version v3
uv run python practica/medidor_calidad_rag.py --version v3 --juez fidelidad          # añade el juez
uv run python practica/medidor_calidad_rag.py --version v3 --comparar-con informes/anterior.json
uv run python practica/medidor_calidad_rag.py --version v3 --exigir-slo              # código de salida 1 si no cumple
```

Aquí hacemos lo mismo, dentro del notebook, con el subconjunto rápido:

💰 **FYI:** 10 llamadas al LLM (≈ US$ 0.005 con un modelo de proveedor; gratis con el modelo local).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.005 con un modelo de proveedor; gratis con el modelo local)
if PUEDE_GASTAR:
    rapido = [p for p in GOLDEN if p["rapido"]]
    informe = evaluar_rag(base, rapido, k=4, version="v3")
    imprimir_resumen(informe, "Medidor de calidad RAG · set rápido")
    print()
    for f in verificar_slo(informe.resumen()):
        print(f"  {'✅' if f['cumple'] else '❌'} {f['metrica']:<22} {f['valor']:>5.0%}   (regla {f['regla']})")
    ruta_json, ruta_md = guardar_informe(informe, "informes", "notebook_set_rapido")
    print(f"\nInforme guardado en {ruta_json} y {ruta_md}")
else:
    print("⚪ Celda de pago desactivada.")

In [ ]:
if PUEDE_GASTAR:
    print(Path("informes/notebook_set_rapido.md").read_text(encoding="utf-8"))

## 🏁 Retos

1. **Tu propio golden set.** Escribe 10 preguntas sobre un documento tuyo (un reglamento, un manual del trabajo) con `secciones` y `claves`, e inclúyelas con `--golden`. ¿Cuántas fallan en recuperación y cuántas en generación?
2. **Una técnica, una falla.** Elige **una** falla del informe (p. ej. las abstenciones indebidas), aplica **una** técnica y vuelve a medir. Escribe en dos líneas qué cambió y si fue señal o ruido.
3. **Prompt v4.** Crea `prompts/rag_respuesta/v4.j2`, anótalo en `prompts/CHANGELOG.md` y compáralo con `v3` usando `--version v4 --comparar-con`.
4. **Rompe tu juez.** Escribe 3 respuestas nuevas (con un error sutil) para `datos/etiquetas_juez.json` y comprueba si tu juez las atrapa.
5. **Mide tu grafo.** Añade 3 servicios al catálogo (`datos/servicios_ti.md`) y sus tripletas curadas; ¿la extracción con el LLM las encuentra?
6. **Pregunta global.** Escribe una pregunta que **solo** un grafo pueda responder bien, y otra que **solo** un vectorial. Explica por qué.

---
# Parte 11 · Cierre

## Lo que aprendiste

**Técnicas avanzadas de recuperación**
1. Un RAG básico falla con preguntas **multi-sección y multi-salto**; subir *k* sube Recall pero hunde Precision.
2. ***Small-to-big***: buscar con fragmentos pequeños, entregar el contexto grande; los **encabezados contextuales** dan tema a fragmentos sueltos.
3. **Descomposición** (separar en subpreguntas, buscar y fusionar con RRF) ataca lo multi-salto; **HyDE** es una apuesta que hay que medir.
4. **GraphRAG** = grafo de conocimiento + RAG: brilla en preguntas **relacionales y globales**, cuesta **extraer, validar y mantener** el grafo, y **no** conviene si tu corpus es pequeño o tus preguntas son directas. Microsoft GraphRAG y LightRAG son las implementaciones de referencia.
5. El **RAG agéntico** deja que el modelo decida qué recuperador usar y cuándo parar: lo verás con agentes.

**Evaluación y alucinaciones**
6. Un RAG se evalúa **por capas**: recuperación (Recall@K, Precision@K, MRR) y generación (dato correcto, citas, "no sé", cifras con respaldo), con un **golden set** versionado.
7. El **diagnóstico por capa** dice *dónde* actuar; **fidelidad ≠ corrección**.
8. Las alucinaciones tienen **causas identificables** (prompt sin "no sé", contexto obsoleto, recuperación pobre, extrapolar…) y cada una su mitigación; algunas están **en los datos**, no en el prompt.
9. Un **juez LLM** es un instrumento que se **calibra** contra etiquetas humanas; uno débil es decoración.
10. **Medir → cambiar → volver a medir**, con un **SLO** que decida si el cambio se despliega.

## ✅ Autoevaluación

1. ¿Por qué Recall@K sube y Precision@K baja al aumentar K? ¿Qué técnica sube Recall sin hundir Precision?
2. ¿Qué dos fallas distintas puede tener una respuesta a una pregunta multi-sección? ¿Cómo sabes cuál ocurrió?
3. ¿Cuándo preferirías *small-to-big* en modo `seccion` y cuándo en modo `ventana`?
4. ¿Por qué la pregunta "si falla X, ¿qué se afecta?" es difícil para un RAG vectorial y fácil para un grafo?
5. Nombra dos costos de GraphRAG que no aparecen en una demo y una situación donde **no** lo usarías.
6. ¿Por qué una arista falsa en el grafo contamina muchas respuestas?
7. Un RAG saca 0 % de alucinaciones y 40 % de "no sé" indebido. ¿Está mejor o peor que otro con 5 % y 5 %? ¿Qué más mirarías?
8. Explica con un ejemplo la diferencia entre fidelidad y corrección.
9. ¿Qué sesgos tiene un juez LLM y cómo compruebas si el tuyo los tiene?
10. ¿Qué hace un SLO en un flujo de CI de un RAG?

## 💰 Resumen de gasto de esta sesión

In [ ]:
total_evaluaciones = sum(i.resumen()["costo_usd"] for i in INFORMES.values())
print(f"Llamadas sueltas: {len(gasto.resultados)} · US$ {gasto.total:.6f}")
print(f"Evaluaciones completas ({len(INFORMES)} informes): US$ {total_evaluaciones:.6f}")
print("(Con el modelo local el costo es 0. Los jueces y las extracciones de tripletas no se suman aquí: súmalos a mano si usas un proveedor.)")

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **Golden set** | Preguntas con respuesta correcta escrita por una persona; la vara de medir |
| **SLI / SLO** | Indicador que se mide / mínimo aceptable que se exige |
| **Recall@K** | Secciones relevantes en el top-K ÷ secciones relevantes que existen |
| **Precision@K** | Fragmentos relevantes del top-K ÷ K |
| **MRR** | 1 ÷ posición del primer resultado relevante |
| **Multi-salto** | Pregunta que exige encadenar datos de varios lugares (y a veces calcular) |
| ***Small-to-big* / *parent-document*** | Buscar con fragmentos pequeños y entregar el contexto grande al que pertenecen |
| **Encabezado contextual** | Anteponer sección o título al fragmento antes de vectorizarlo |
| **Descomposición** | Separar una pregunta en subpreguntas de un solo tema |
| **RRF** | Fusión por rango recíproco: combina varias listas sin comparar sus puntajes |
| **HyDE** | Buscar con una respuesta hipotética escrita por el LLM |
| **Grafo de conocimiento** | Red de entidades (nodos) y relaciones (aristas) |
| **Tripleta** | Un hecho `(sujeto, relación, objeto)` |
| **GraphRAG** | RAG que usa un grafo de conocimiento (búsqueda local y global) |
| **Búsqueda local / global** | Entidades de la pregunta y sus vecinos / resúmenes por comunidad del grafo |
| **Comunidad** | Grupo de nodos muy conectados entre sí (algoritmo de Leiden) |
| **RAG agéntico** | RAG donde un agente decide qué buscar y cuándo parar |
| **Fidelidad (*groundedness*)** | Todo lo afirmado está respaldado por el contexto |
| **Corrección** | Lo afirmado es verdad (coincide con lo esperado) |
| **Alucinación** | Respuesta plausible pero no respaldada o falsa |
| **Abstención** | Responder "no sé" cuando el contexto no alcanza |
| **LLM-as-judge** | Usar un modelo, con rúbrica, para evaluar respuestas |
| **Kappa de Cohen** | Acuerdo entre evaluadores descontando el azar |
| **Regresión** | Que un cambio empeore algo que ya funcionaba; se detecta volviendo a medir |

## ➡️ Próxima semana
Con un RAG medido y mejorado tienes la base para **dar al modelo herramientas y capacidad de decidir**: los **agentes**. Tu golden set y tu Medidor de calidad serán la prueba de regresión del hito **H3**.